### Week 6, Day 2

We're about to create and use our own MCP Server and MCP Client!

It's pretty simple, but it's not super-simple. The excitment around MCP is about how easy it is to share and use other MCP Servers - making our own does involve a bit of work.

Let's review some python code made mostly by a hard-working Engineering Team:

accounts.py

In [1]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
from IPython.display import display, Markdown

load_dotenv(override=True)

True

In [2]:
from accounts import Account

In [3]:
account = Account.get("Ed")
account

Account(name='ed', balance=10000.0, strategy='', holdings={}, transactions=[], portfolio_value_time_series=[])

In [4]:
account.buy_shares("AMZN", 3, "Because this bookstore website looks promising")

'Completed. Latest details:\n{"name": "ed", "balance": 9762.526, "strategy": "", "holdings": {"AMZN": 3}, "transactions": [{"symbol": "AMZN", "quantity": 3, "price": 79.158, "timestamp": "2026-02-24 16:00:34", "rationale": "Because this bookstore website looks promising"}], "portfolio_value_time_series": [["2026-02-24 16:00:34", 9816.526]], "total_portfolio_value": 9816.526, "total_profit_loss": -183.47400000000016}'

In [5]:
account.report()

'{"name": "ed", "balance": 9762.526, "strategy": "", "holdings": {"AMZN": 3}, "transactions": [{"symbol": "AMZN", "quantity": 3, "price": 79.158, "timestamp": "2026-02-24 16:00:34", "rationale": "Because this bookstore website looks promising"}], "portfolio_value_time_series": [["2026-02-24 16:00:34", 9816.526], ["2026-02-24 16:00:37", 9861.526]], "total_portfolio_value": 9861.526, "total_profit_loss": -138.47400000000016}'

In [6]:
account.list_transactions()

[{'symbol': 'AMZN',
  'quantity': 3,
  'price': 79.158,
  'timestamp': '2026-02-24 16:00:34',
  'rationale': 'Because this bookstore website looks promising'}]

### Now we write an MCP server and use it directly!

In [7]:
# Now let's use our accounts server as an MCP server

params = {"command": "uv", "args": ["run", "accounts_server.py"]}
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as server:
    mcp_tools = await server.list_tools()


In [8]:
mcp_tools

[Tool(name='get_balance', title=None, description='Get the cash balance of the given account name.\n\n    Args:\n        name: The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_balanceArguments', 'type': 'object'}, outputSchema={'properties': {'result': {'title': 'Result', 'type': 'number'}}, 'required': ['result'], 'title': 'get_balanceOutput', 'type': 'object'}, icons=None, annotations=None, meta=None),
 Tool(name='get_holdings', title=None, description='Get the holdings of the given account name.\n\n    Args:\n        name: The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_holdingsArguments', 'type': 'object'}, outputSchema={'additionalProperties': {'type': 'integer'}, 'title': 'get_holdingsDictOutput', 'type': 'object'}, icons=None, annotations=None, meta=None),
 Tool(name='buy_shares', 

In [9]:
instructions = "You are able to manage an account for a client, and answer questions about the account."
request = "My name is Ed and my account is under the name Ed. What's my balance and my holdings?"
model = "gpt-4.1-mini"

In [10]:

async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="account_manager", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("account_manager"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))


Ed, your account balance is $9,762.53. You currently hold 3 shares of Amazon (AMZN). Is there anything you would like to do with your account?

### Now let's build our own MCP Client

In [11]:
from accounts_client import get_accounts_tools_openai, read_accounts_resource, list_accounts_tools

mcp_tools = await list_accounts_tools()
print(mcp_tools)
openai_tools = await get_accounts_tools_openai()
print(openai_tools)

[Tool(name='get_balance', title=None, description='Get the cash balance of the given account name.\n\n    Args:\n        name: The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_balanceArguments', 'type': 'object'}, outputSchema={'properties': {'result': {'title': 'Result', 'type': 'number'}}, 'required': ['result'], 'title': 'get_balanceOutput', 'type': 'object'}, icons=None, annotations=None, meta=None), Tool(name='get_holdings', title=None, description='Get the holdings of the given account name.\n\n    Args:\n        name: The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_holdingsArguments', 'type': 'object'}, outputSchema={'additionalProperties': {'type': 'integer'}, 'title': 'get_holdingsDictOutput', 'type': 'object'}, icons=None, annotations=None, meta=None), Tool(name='buy_shares', ti

In [12]:
request = "My name is Ed and my account is under the name Ed. What's my balance?"

with trace("account_mcp_client"):
    agent = Agent(name="account_manager", instructions=instructions, model=model, tools=openai_tools)
    result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

The cash balance in your account is $9,762.53. Let me know if you need any more information or assistance with your account.

In [13]:
context = await read_accounts_resource("ed")
print(context)

{"name": "ed", "balance": 9762.526, "strategy": "", "holdings": {"AMZN": 3}, "transactions": [{"symbol": "AMZN", "quantity": 3, "price": 79.158, "timestamp": "2026-02-24 16:00:34", "rationale": "Because this bookstore website looks promising"}], "portfolio_value_time_series": [["2026-02-24 16:00:34", 9816.526], ["2026-02-24 16:00:37", 9861.526], ["2026-02-24 16:25:45", 9792.526]], "total_portfolio_value": 9792.526, "total_profit_loss": -207.47400000000016}


In [14]:
from accounts import Account
Account.get("ed").report()

'{"name": "ed", "balance": 9762.526, "strategy": "", "holdings": {"AMZN": 3}, "transactions": [{"symbol": "AMZN", "quantity": 3, "price": 79.158, "timestamp": "2026-02-24 16:00:34", "rationale": "Because this bookstore website looks promising"}], "portfolio_value_time_series": [["2026-02-24 16:00:34", 9816.526], ["2026-02-24 16:00:37", 9861.526], ["2026-02-24 16:25:45", 9792.526], ["2026-02-24 16:25:46", 9924.526]], "total_portfolio_value": 9924.526, "total_profit_loss": -75.47400000000016}'

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercises</h2>
            <span style="color:#ff7800;">Make your own MCP Server! Make a simple function to return the current Date, and expose it as a tool so that an Agent can tell you today's date.<br/>Harder optional exercise: then make an MCP Client, and use a native OpenAI call (without the Agents SDK) to use your tool via your client.
            </span>
        </td>
    </tr>
</table>

### Exercise: Basic — MCP Server with a `get_current_date` tool

The server lives in `date_server.py`. It exposes a single tool that returns the current date and time. We use it via `MCPServerStdio`, exactly like the accounts server above.

In [15]:
# Inspect the tools exposed by our date server
date_params = {"command": "uv", "args": ["run", "date_server.py"]}
async with MCPServerStdio(params=date_params, client_session_timeout_seconds=30) as server:
    date_tools = await server.list_tools()

date_tools

[Tool(name='get_current_date', title=None, description='Get the current date and time.\n\n    Returns the current date and time as a formatted string.\n    ', inputSchema={'properties': {}, 'title': 'get_current_dateArguments', 'type': 'object'}, outputSchema={'properties': {'result': {'title': 'Result', 'type': 'string'}}, 'required': ['result'], 'title': 'get_current_dateOutput', 'type': 'object'}, icons=None, annotations=None, meta=None)]

In [16]:
# Use the date server with an Agent via the Agents SDK
async with MCPServerStdio(params=date_params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(
        name="date_agent",
        instructions="You are a helpful assistant. Use your tools to answer questions accurately.",
        model="gpt-4.1-mini",
        mcp_servers=[mcp_server],
    )
    with trace("date_agent"):
        result = await Runner.run(agent, "What is today's date?")
    display(Markdown(result.final_output))

Today's date is Tuesday, February 24, 2026.

### Exercise: Harder — Native OpenAI call via a custom MCP Client

`date_client.py` implements:
- `list_date_tools()` — connects to the MCP server and lists its tools
- `call_date_tool()` — calls a specific tool by name
- `ask_with_date_tools()` — a **manual agentic loop** using the raw `openai` library (no Agents SDK): sends the question, handles tool-call responses, feeds results back, and loops until the model gives a final answer

In [17]:
from date_client import ask_with_date_tools, list_date_tools

# Confirm our client can see the tool
tools = await list_date_tools()
print("Tools available:", [t.name for t in tools])

Tools available: ['get_current_date']


In [18]:
# Native OpenAI call — no Agents SDK, just the raw openai library + our MCP client
answer = await ask_with_date_tools("What is today's date and day of the week?")
display(Markdown(answer))

Today's date is Tuesday, February 24, 2026, and the day of the week is Tuesday.